# Ensemble Affinity Analysis (On-the-fly)

This notebook computes PRP and RA summaries in memory from merged benchmark files.

No summary CSVs are created and figures are shown inline only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px

import sys
scripts_dir = (Path.cwd().resolve().parent / "scripts") if Path.cwd().name == "notebooks" else (Path.cwd().resolve() / "scripts")
if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

from run_affinity_adjudication import (
    # compute_pair_stats,
    normalize_abstract_index,
    affinity_level_fixed,
 )


In [ ]:
def prepare_data(df):
    df["LLM_Affinity"] = pd.to_numeric(df["LLM_Affinity"], errors="coerce")
    df = df[df["LLM_Affinity"].notna()].copy()
    df["Abstract_Index_Norm"] = df["Abstract_Index"]
    # df["Target_ID_Norm"] = df["RA2025_ID"].apply(_normalize_target_id)
    df["affinity_level"] = df["LLM_Affinity"].apply(affinity_level_fixed)

    return df

def compute_pair_stats(merged_df: pd.DataFrame, target_id_col: str, target_text_col: str, other_cols: list = None) -> pd.DataFrame:
    """Compute pair-level statistics from merged affinity DataFrame."""
    base_cols = [
        "Abstract_Index",
        "Abstract_Index_Norm",
        "Document Title",
        target_id_col,
        target_text_col,
    ]
    if other_cols:
        base_cols.extend(other_cols)

    merged_df['affinity_level'] = merged_df['LLM_Affinity'].apply(affinity_level_fixed)
    grouped = (
        merged_df.groupby(base_cols, dropna=False, as_index=False)
        .agg(
            n_models=("Model_Slug", "nunique"),
            affinity_mean=("LLM_Affinity", "mean"),
            affinity_std=("LLM_Affinity", "std"),
            affinity_min=("LLM_Affinity", "min"),
            affinity_max=("LLM_Affinity", "max"),
            n_affinity_levels=("affinity_level", "nunique"),
        )
    )
    
    grouped["affinity_std"] = grouped["affinity_std"].fillna(0.0)
    grouped["affinity_range"] = grouped["affinity_max"] - grouped["affinity_min"]

    # Compute per-level model agreement once, then keep only the maximum ratio per pair.
    if 'affinity_level' not in base_cols:
        base_cols_with_level = base_cols + ["affinity_level"]
        base_cols_without_level = base_cols
    else:
        base_cols_with_level = base_cols
        base_cols_without_level = base_cols.copy()
        base_cols_without_level.remove("affinity_level")
    
    df_per_affinity_level = (
        merged_df.groupby(base_cols_with_level, dropna=False)["Model_Slug"]
        .nunique()
        .reset_index(name="n_models_per_affinity")
    )

    n_models_by_pair = merged_df.groupby(base_cols_without_level, dropna=False)["Model_Slug"].nunique()
    pair_keys = pd.MultiIndex.from_frame(df_per_affinity_level[base_cols_without_level])
    df_per_affinity_level["n_models"] = pair_keys.map(n_models_by_pair.to_dict())
    df_per_affinity_level["agreement_ratio"] = (
        df_per_affinity_level["n_models_per_affinity"] / df_per_affinity_level["n_models"]
    )
    agreement_stats = (
        df_per_affinity_level.groupby(base_cols, dropna=False, as_index=False)["agreement_ratio"]
        .agg(
            agreement_mean="mean",
            agreement_std="std",
            agreement_min="min",
            agreement_max="max",
        )
    )
    agreement_stats["agreement_std"] = agreement_stats["agreement_std"].fillna(0.0)
    grouped = grouped.merge(agreement_stats, on=base_cols, how="left")

    return grouped

In [ ]:
# Set run_id explicitly, or keep None to use latest run folder.
ignore_models = ['smollm2-1.7b', 'gemma3-4b', 'phi3-3.8b']
run_id =None
# run_id = "20260401_082538"
# run_id = "20260406_211752"
# run_id = "20260413_151017"
# run_id = "20260421_184019"

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
bench_root = repo_root / "data" / "results" / "affinity_benchmark"

if run_id is None:
    run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
    if not run_dirs:
        raise FileNotFoundError(f"No benchmark runs found under {bench_root}")
    run_dir = run_dirs[-1]
else:
    run_dir = bench_root / run_id

prp_path = run_dir / "merged_prp_affinities.csv"
ra_path = run_dir / "merged_ra_affinities.csv"

if not prp_path.exists() :
    raise FileNotFoundError(
        "Merged files not found. Run scripts/run_postprocessing.py first for this run_id."
    )
prp_df = pd.read_csv(prp_path)
prp_df = prp_df[~prp_df['Model_Slug'].isin(ignore_models)]

print("Run directory:", run_dir)
print("PRP rows:", len(prp_df), "| Models:", prp_df['Model_Slug'].nunique())
if ra_path.exists():
    ra_df = pd.read_csv(ra_path)
    ra_df = ra_df[~ra_df['Model_Slug'].isin(ignore_models)]
    print("RA rows:", len(ra_df), "| Models:", ra_df['Model_Slug'].nunique())




# prp_df['Abstract_Index_Norm'] = prp_df['Abstract_Index'].apply(normalize_abstract_index)
# prp_df['affinity_level'] = prp_df['LLM_Affinity'].apply(affinity_level_fixed)
prp_df = prepare_data(prp_df)
prp_agreement = compute_pair_stats(prp_df, target_id_col="PRP_Name", target_text_col="PRP_Description")

ra_df = prepare_data(ra_df)
ra_agreement = compute_pair_stats(ra_df, target_id_col="RA2025_ID", target_text_col="RA_Question")
ra_agreement['agreement'] = ra_agreement['agreement_mean'] == 1

# ra_agreement["affinity_level"] = ra_agreement["affinity_mean"].apply(affinity_level_fixed)
ra_agreement_per_affinity = compute_pair_stats(ra_df, target_id_col="RA2025_ID", target_text_col="RA_Question", other_cols=["affinity_level"]) 
ra_agreement_per_affinity['agreement'] = ra_agreement_per_affinity['agreement_mean'] == 1
# ra_df['Abstract_Index_Norm'] = ra_df['Abstract_Index'].apply(normalize_abstract_index)
# ra_df['affinity_level'] = ra_df['LLM_Affinity'].apply(affinity_level_fixed)





In [ ]:
mapping_path = repo_root / 'data' / 'processed' / 'ra_grouping_ra2025.csv'
ra_grouping_df = pd.read_csv(mapping_path)

required_cols = {'RA2025', 'Grouping'}
missing_cols = required_cols - set(ra_grouping_df.columns)
if missing_cols:
    raise ValueError(f'Missing required columns in mapping file: {missing_cols}')

ra_to_prp_map = (
    ra_grouping_df[['RA2025', 'Grouping']]
    .dropna(subset=['RA2025', 'Grouping'])
    .drop_duplicates()
    .rename(columns={'RA2025': 'RA2025_ID', 'Grouping': 'PRP_Name'})
)
ra_to_prp_map['RA2025_ID'] = pd.to_numeric(ra_to_prp_map['RA2025_ID'], errors='coerce').astype('Int64')

print('Mapping file:', mapping_path)
print('Mapping rows:', len(ra_to_prp_map))
display(ra_to_prp_map.head(10))
ra_df = ra_df.merge(
    ra_to_prp_map,
    on='RA2025_ID',
    how='left'
 )

ra_agreement = ra_agreement.merge(
    ra_to_prp_map,
    on='RA2025_ID',
    how='left'
)
ra_agreement_per_affinity = ra_agreement_per_affinity.merge(
    ra_to_prp_map,
    on='RA2025_ID',
    how='left'
)


In [ ]:
# Quick coverage checks
display(prp_df[['Model_Slug', 'Abstract_Index']].drop_duplicates().groupby('Model_Slug').size().rename('n_abstracts').reset_index())
display(ra_df[['Model_Slug', 'Abstract_Index']].drop_duplicates().groupby('Model_Slug').size().rename('n_abstracts').reset_index())

# General

In [ ]:
fig = px.box(
    prp_df,
    x='Model_Slug',
    y='LLM_Affinity',
    # color = 'PRP_Name',
    # points = 'all',
    title='Affinity Distribution by PRP across models',
 )
fig.update_layout(xaxis_title='Model', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

fig = px.box(
    ra_df,
    x='Model_Slug',
    y='LLM_Affinity',
    title='RA Affinity Distribution by Model'
 )
fig.update_layout(xaxis_title='Model', yaxis_title='LLM Affinity')
fig.update_xaxes(tickangle=45)
fig.show()

# PRP Affinity level
Affinity level of an **abstract** to each **primary research programme** (PRP)

In [ ]:
fig = px.box(
    prp_df,
    color='Model_Slug',
    y='LLM_Affinity',
    x = 'PRP_Name',
    # points = 'all',
    title='Affinity Distribution by PRP across models',
 )
# fig.update_layout(xaxis_title='PRP', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
fig = px.line(
    prp_df[prp_df['Abstract_Index'].isin([1,8,20, 19, 30])], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    # points='all',
    # hover_data='PRP_Name',
    # facet_row='PRP_Name',

    facet_col='Abstract_Index',
    color = 'PRP_Name',
    x = 'Model_Slug',
    title=f'PRP Affinity Distribution by Abstract',
    markers = True
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# # Patch px.box so every new box figure gets affinity threshold guides
# if not hasattr(px, "_original_box"):
#     px._original_box = px.box

# def _box_with_affinity_guides(*args, **kwargs):
#     fig = px._original_box(*args, **kwargs)

#     # horizontal separators
#     fig.add_hline(y=40, line_dash="dash", line_color="gray")
#     fig.add_hline(y=70, line_dash="dash", line_color="gray")

#     # optional background bands
#     fig.add_hrect(y0=0, y1=40, fillcolor="red", opacity=0.06, line_width=0)
#     fig.add_hrect(y0=40, y1=70, fillcolor="orange", opacity=0.06, line_width=0)
#     fig.add_hrect(y0=70, y1=100, fillcolor="green", opacity=0.06, line_width=0)

#     # range labels
#     fig.add_annotation(x=1.11, y=20, xref="paper", yref="y", text="Low ", showarrow=False)
#     fig.add_annotation(x=1.18, y=55, xref="paper", yref="y", text="Medium ", showarrow=False)
#     fig.add_annotation(x=1.11, y=85, xref="paper", yref="y", text="High", showarrow=False)

#     return fig

# px.box = _box_with_affinity_guides
# fig = px.box(
#     prp_df[prp_df['Abstract_Index'] == 8], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
#     y='LLM_Affinity',
#     points='all',
#     hover_data='PRP_Name',
#     x = 'PRP_Name',
#     color = 'PRP_Name',
#     title=f'PRP Affinity Distribution by PRP for Abstract 8'
#  )
# fig.update_layout(xaxis_title='PRP Name', yaxis_title='LLM Affinity' ,showlegend=False)
# # fig.update_xaxes(tickangle=45)
# fig.update_layout(width=500, height=500)
# fig.show()

## PRP Summaries

In [ ]:
prp_summary = (
    prp_df.groupby(['Model_Slug', 'PRP_Name'], as_index=False)['LLM_Affinity']
    .agg(mean_affinity='mean', std_affinity='std', n='count')
)
display(prp_summary.head(20))

In [ ]:
prp_heat = prp_summary.pivot(index='Model_Slug', columns='PRP_Name', values='mean_affinity').sort_index()
fig = px.imshow(
    prp_heat,
    text_auto='.1f',
    color_continuous_scale='Viridis',
    title='Mean PRP Affinity (Model x PRP)',
    aspect='auto'
 )
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Model')
fig.show()

## RA Summaries

In [ ]:
ra_df

In [ ]:
fig = px.box(#prp_df[prp_df['Abstract_Index'].isin([1,8,20,19])]
    ra_df,
    # ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    color='affinity_level',
    # facet_col = 'Abstract_Index',
    # points='all',
    hover_data = 'Model_Slug',
    # facet_row='PRP_Name',
    # color = 'PRP_Name',
    # title=f'RA Affinity Distribution by Abstract'
 )
fig.update_layout(xaxis_title='RA question', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
fig = px.scatter(#prp_df[prp_df['Abstract_Index'].isin([1,8,20,19])]
    ra_df[ra_df['Abstract_Index'].isin([1,8,20,19])],
    # ra_df,
    x='RA2025_ID',
    y='LLM_Affinity',
    color='Model_Slug',
    facet_col = 'Abstract_Index',
    # points='all',
    hover_data = 'Model_Slug',
    # facet_row='PRP_Name',
    # color = 'PRP_Name',
    # title=f'RA Affinity Distribution by Abstract'
 )
fig.update_layout(xaxis_title='RA question', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

# Agreement

In [ ]:
# top_disaggrement_ids = [1,8,20, 19]
# top_disaggrement_ids = [58]
top_disaggrement_ids = (prp_agreement[prp_agreement['agreement_mean']<1]).nlargest(5, 'affinity_range')['Abstract_Index']
low_disagreements_ids = (prp_agreement[prp_agreement['agreement_mean']==1]).nsmallest(5, 'affinity_range')['Abstract_Index']
# low_ra_disaggrement_ids = (ra_adjudication_df[ra_adjudication_df['agreement_mean']==1]).nsmallest(12, 'Affinity_Range')['Abstract_Index']
fig = px.box(
    prp_df[prp_df['Abstract_Index'].isin(top_disaggrement_ids)], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    points='all',
    hover_data='PRP_Name',
    # facet_row='PRP_Name',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    # x = 'Model_Slug',
    title=f'PRP Affinity Distribution by Abstract'
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()

fig = px.box(
    prp_df[prp_df['Abstract_Index'].isin(low_disagreements_ids)], #ra_df[ra_df['Abstract_Index'].isin([7,8,9])],
    # x='Abstract_Index',
    y='LLM_Affinity',
    # color='Model_Slug',
    points='all',
    hover_data='PRP_Name',
    # facet_row='PRP_Name',
    facet_col='Abstract_Index',
    color = 'PRP_Name',
    # x = 'Model_Slug',
    title=f'PRP Affinity Distribution by Abstract'
 )
# fig.update_xaxes(range=[0.5,10.5])
fig.update_layout(xaxis_title='Abstract Index', yaxis_title='LLM Affinity')
# fig.update_xaxes(tickangle=45)
fig.show()



In [ ]:
# overall_agreement

In [ ]:
# Heatmap: agreement ratio by Abstract and PRP

overall_agreement = (prp_agreement.agreement_mean == 1).sum()/(prp_agreement.agreement_mean).count()

prp_agreement_heat = (
    prp_agreement.pivot(index='Abstract_Index', columns='PRP_Name', values='agreement_mean')
    .sort_index()
)


fig = px.imshow(
    prp_agreement_heat,
    text_auto='.2f',
    zmin=prp_agreement_heat.min().min(),
    zmax=prp_agreement_heat.max().max(),
    color_continuous_scale='RdYlGn',
    title=f'Affinity Level Mean Agreement Ratio Across Models (Abstract x PRP): {overall_agreement:.2%}',
    labels={'color': 'Agreement ratio'},
    aspect='auto',
)
fig.update_layout(width=800, height=600)
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Abstract Index')


fig.show()

In [ ]:
ra_agreement_per_affinity.sort_values('RA2025_ID', ascending=False)

In [ ]:
# ra_agreement_per_affinity['agreement_mean'] = ra_agreement_per_affinity['agreement_mean'].astype(float)

fig = px.scatter(
    ra_agreement_per_affinity,
    x='RA2025_ID',
    y='Abstract_Index',
    color='agreement_mean',
    # nbinsx = 100, #ra_agreement_per_affinity.RA2025_ID.max(),
    # nbinsy= 100, #ra_agreement_per_affinity.Abstract_Index.max(),
    # histfunc='max',
    # symbol = '+',
    # marker=dict(size=12, opacity=0.8),
    facet_col='affinity_level',
    # color_continuous_scale='blues',
    category_orders = {'affinity_level': ['Low', 'Moderate', 'High']},
    # title='Agreement by RA question and Abstract (per affinity level)',
    labels={'agreement_mean':'agreement', 'RA2025_ID':'RA Question', 'Abstract_Index':'Abstract Index'},
    # histfunc = 'mean',
    # histnorm = 'probability',
    # marginal_y  = 'histogram',
    # marker_symbol='square',
    marginal_x = 'histogram',
    # marginal_y = 'histogram',
    # aspect='auto'
)
# fig.update_layout
# fig.update_layout(xaxis_title='RA Question', yaxis_title='Abstract Index')
fig.update_layout(width=1000, height=600)
fig.update_traces(marker=dict(size=4,symbol="square",),selector=dict(mode='markers'))
# fig.update_yaxes(autorange='reversed')
fig.show()

In [ ]:
ra_agreement_per_affinity

In [ ]:
fig = px.histogram(
    ra_agreement_per_affinity,
    x = 'Abstract_Index',
    # y = 'agreement_mean',
    
    color='affinity_level',
    facet_col='agreement',
    barmode ='overlay',
    # histfunc ='count',
    # category_orders = {'affinity_level': ['Low', 'Moderate', 'High']},
    category_orders = {'affinity_level': ['Low', 'Moderate', 'High']},
    # nbins=20,
    # title='Distribution of agreement ratios by affinity level',
    # histnorm='relative',

)
fig.update_layout(width=700, height=400)
# fig.update_layout(barmode=')
# fig.update_traces(opacity=0.75)
fig.show()

In [ ]:
ra_agreement_per_affinity.groupby(['affinity_level','agreement'])['agreement_mean'].describe()

In [ ]:
ra_agreement_per_affinity.groupby('affinity_level')['agreement_mean']

fig = px.histogram(
    ra_agreement_per_affinity,
    color='agreement',
    x='affinity_level',
    barmode ='group',
    # category_orders = {'affinity_level': ['Low', 'Moderate', 'High']},
    category_orders = {'affinity_level': ['Low', 'Moderate', 'High']},
    # nbins=20,
    # title='Distribution of agreement ratios by affinity level',
    histnorm='percent',

)
fig.update_layout(width=400, height=400)
# fig.update_layout(barmode=')
# fig.update_traces(opacity=0.75)
fig.show()

In [ ]:
ra_agreement

In [ ]:
ra_df_disagreement

In [ ]:
disagreement_pairs = ra_agreement_per_affinity.loc[
    ra_agreement_per_affinity["agreement"] == 0,
    ["Abstract_Index", "RA2025_ID"]
].drop_duplicates()



ra_df_disagreement = ra_df.copy()

ra_df_disagreement = ra_df.merge(disagreement_pairs, on=["Abstract_Index", "RA2025_ID"], how="inner")
ra_df_disagreement = ra_df_disagreement.merge(
    ra_agreement_per_affinity[['Abstract_Index', 'RA2025_ID', 'affinity_mean']],
    on=['Abstract_Index', 'RA2025_ID'],
    how='left'
)
ra_df_disagreement['LLM_affinity_deviation'] = (ra_df_disagreement['LLM_Affinity'] - ra_df_disagreement['affinity_mean'])


In [ ]:
ra_df_disagreement.groupby('Model_Slug')['LLM_affinity_deviation'].describe()

In [ ]:
fig = px.box(
    ra_df_disagreement,
    # color='RA2025_ID',
    x='LLM_affinity_deviation',
    y='Model_Slug',
    # symbol = 'Abstract_Index',
    # points='all',
    # barmode ='overlay',
    # color = ''
    hover_data = 'Model_Slug',
    # facet_row='PRP_Name',
    # color = 'PRP_Name',
    title=f'RA Affinity Deviation from Mean by Abstract'
 )
fig.update_layout(width=600, height=400)
# fig.update_layout(barmode=')
# fig.update_traces(opacity=0.75)
fig.show()

In [ ]:
# ra_df_disagreement.groupby('Model_Slug')['LLM_affinity_deviation'].describe()

In [ ]:
fig = px.bar(ra_df_disagreement.groupby('Model_Slug')['LLM_affinity_deviation'].describe().reset_index(), y ='Model_Slug',x = 'std')
fig.update_layout(width=600, height=400)

In [ ]:
# px.box(
#     ra_df_disagreement,
#     x='RA2025_ID',
#     y='LLM_affinity_deviation',
#     facet_col='Model_Slug',
#     # symbol = 'Abstract_Index',
#     # points='all',
#     # hover_data = 'Model_Slug',
#     # facet_row='PRP_Name',
#     # color = 'PRP_Name',
#     title=f'RA Affinity Deviation from Mean by Abstract'
#  )

In [ ]:
ra_agreement_per_affinity

In [ ]:
# Select the row with maximum agreement_mean per (Abstract_Index, PRP_Name)
idx = (
    ra_agreement_per_affinity
    .groupby(['Abstract_Index', 'PRP_Name'], dropna=False)['agreement_mean']
    .idxmax()
)
# idx may contain NaN if groups empty; dropna
idx = idx.dropna().astype(int)
prp_agreement_per_affinity_from_ra = (
    ra_agreement_per_affinity.loc[idx, ['Abstract_Index', 'PRP_Name', 'agreement_mean', 'affinity_mean']]
    .rename(columns={'agreement_mean': 'agreement_max'})
    .reset_index(drop=True)
)
# Now `affinity_mean` corresponds to the row that maximises `agreement_mean` in each group
prp_agreement_per_affinity_from_ra['affinity_level'] = prp_agreement_per_affinity_from_ra['affinity_mean'].apply(affinity_level_fixed)
prp_agreement_per_affinity_from_ra['agreement'] = prp_agreement_per_affinity_from_ra['agreement_max'] == 1

# Density heatmap showing agreement_max by PRP (faceted by affinity level)
fig = px.density_heatmap(
    prp_agreement_per_affinity_from_ra,
    x='PRP_Name',
    y='Abstract_Index',
    z='agreement_max',
    nbinsx=100,
    nbinsy=100,
    histfunc='avg',
    facet_col='affinity_level',
    color_continuous_scale='blues',
    category_orders={'affinity_level': ['Low', 'Moderate', 'High']},
    labels={'agreement_max': 'agreement'},
)
fig.update_layout(width=800, height=600, showlegend=False)
fig.update_yaxes(autorange='reversed')
fig.show()

In [ ]:
prp_agreement_per_affinity_from_ra

In [ ]:

fig = px.histogram(
    prp_agreement_per_affinity_from_ra,
    x = 'PRP_Name',
    # y = 'agreement_mean',
    color='affinity_level',
    facet_col='agreement',
    barmode ='overlay',
    # category_orders = {'affinity_level': ['Low', 'Moderate', 'High']},
    category_orders = {'affinity_level': ['Low', 'Moderate', 'High']},
    # nbins=20,
    # title='Distribution of agreement ratios by affinity level',
    # histnorm='relative',

)
fig.update_layout(width=700, height=400)
# fig.update_layout(barmode=')
# fig.update_traces(opacity=0.75)
fig.show()

In [ ]:
fig = px.box(
    ra_df_disagreement.groupby('Model_Slug')['LLM_affinity_deviation'].describe().reset_index(),
    x='Model_Slug',
    y='50%',
    title='Distribution of LLM Affinity Deviation by Model',
    error_y='std'
)
fig.update_layout(xaxis_title='Model', yaxis_title='Median Affinity Deviation')
fig.update_xaxes(tickangle=45)
fig.show()# ra_agreement_by_prp_heat = ra_agreement.groupby(['Abstract_Index','PRP_Name']).agg(agreement = ('agreement_mean','max')).reset_index()
# ra_agreement_by_prp_heat =ra_agreement_by_prp_heat.pivot(index='Abstract_Index', columns='PRP_Name', values='agreement').sort_index()

# fig = px.imshow(
#     ra_agreement_by_prp_heat,
#     text_auto='.2f',
#     zmin=ra_agreement_by_prp_heat.min().min(),
#     zmax=ra_agreement_by_prp_heat.max().max(),
#     color_continuous_scale='RdYlGn',
#     title=f'Affinity Level Mean Agreement Ratio Across Models (Abstract x PRP): {overall_agreement:.2%}',
#     # labels={'color': 'Agreement ratio'},
#     aspect='auto',
# )
# fig.update_layout(width=800, height=600)
# fig.update_layout(xaxis_title='PRP Name', yaxis_title='Abstract Index')


# fig.show()

In [ ]:
# ra_agreement[(ra_agreement['Abstract_Index']==72) & (ra_agreement['PRP_Name']=='CROF')]

In [ ]:
# (Replaced duplicate/messy definition) PRP aggregation computed above using idxmax to pick affinity matching max agreement.

In [ ]:
ra_agreement['agreement'] = (ra_agreement['agreement_mean'] == 1).astype(int)
ra_agreement.groupby(['PRP_Name'])['agreement'].sum() / ra_agreement.groupby(['PRP_Name'])['agreement'].count() *100

In [ ]:
ra_agreement.groupby(['PRP_Name', 'agreement_mean'])['Abstract_Index'].count().reset_index()

## Coherence Analysis: Model Agreement on Affinity Levels

Analyze whether all models assign similar affinity scores to the same questions/abstracts.

## Coherence Logic: Coefficient of Variation (CV)

The coefficient of variation measures how dispersed the model affinities are relative to their average value.  
For each abstract-target unit, we collect the affinity scores from all models and compute:

$$
\mu = \frac{1}{n}\sum_{i=1}^{n} x_i
$$

$$
\sigma = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\mu)^2}
$$

$$
CV = \frac{\sigma}{\mu + \varepsilon}
$$

where:

- $x_i$ is the affinity score assigned by model $i$
- $n$ is the number of models
- $\mu$ is the mean affinity
- $\sigma$ is the sample standard deviation
- $\varepsilon$ is a small constant added to avoid division by zero

Interpretation:

- Low CV means the models are in close agreement
- High CV means the models disagree more strongly
- CV can become unstable when the mean affinity is very small, so low-mean cases should be interpreted with caution

In [ ]:
def compute_unit_coherency(df, target_col, eps=1e-8, low_mean_threshold=5.0):
    required_cols = {'Abstract_Index', 'Model_Slug', target_col, 'LLM_Affinity'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    unit_stats = (
        df.groupby(['Abstract_Index', target_col], as_index=False)['LLM_Affinity']
        .agg(
            mean_affinity='mean',
            std_affinity='std',
            min_affinity='min',
            max_affinity='max',
            n_models='count',
            mad_affinity=lambda s: float(np.mean(np.abs(s - s.mean()))),
        )
    )
    unit_stats['cv'] = unit_stats['std_affinity'] / (unit_stats['mean_affinity'] + eps)
    unit_stats.loc[unit_stats['n_models'] < 2, 'cv'] = np.nan
    unit_stats['range'] = unit_stats['max_affinity'] - unit_stats['min_affinity']
    unit_stats['Affinity_Level'] = unit_stats['mean_affinity'].apply(affinity_level_fixed)
    unit_stats['cv_low_mean_flag'] = unit_stats['mean_affinity'] < float(low_mean_threshold)
    unit_stats['singleton_flag'] = unit_stats['n_models'] < 2
    return unit_stats


def summarize_coherency_by_level(unit_stats):
    level_order = pd.CategoricalDtype(categories=['Low', 'Moderate', 'High'], ordered=True)
    out = unit_stats.copy()
    out['Affinity_Level'] = out['Affinity_Level'].astype(level_order)

    level_summary = (
        out.dropna(subset=['Affinity_Level'])
        .groupby('Affinity_Level', as_index=False, observed=False)
        .agg(
            n_units=('Affinity_Level', 'count'),
            mean_cv=('cv', 'mean'),
            median_cv=('cv', 'median'),
            p90_cv=('cv', lambda s: float(np.percentile(s.dropna(), 90)) if s.notna().any() else np.nan),
            mean_mad=('mad_affinity', 'mean'),
            median_mad=('mad_affinity', 'median'),
            mean_range=('range', 'mean'),
            pct_low_mean_flag=('cv_low_mean_flag', 'mean'),
            pct_singleton=('singleton_flag', 'mean'),
        )
        .sort_values('Affinity_Level')
    )
    level_summary['pct_low_mean_flag'] = 100.0 * level_summary['pct_low_mean_flag']
    level_summary['pct_singleton'] = 100.0 * level_summary['pct_singleton']
    return level_summary


def compute_pairwise_disagreement_by_level(df, target_col):
    records = []
    for (abstract_idx, target_id), grp in df.groupby(['Abstract_Index', target_col]):
        grp2 = grp[['Model_Slug', 'LLM_Affinity']].dropna()
        if grp2['Model_Slug'].nunique() < 2:
            continue

        level = affinity_level_fixed(grp2['LLM_Affinity'].mean())
        rows = grp2.sort_values('Model_Slug')[['Model_Slug', 'LLM_Affinity']].to_records(index=False)

        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                m1, a1 = rows[i]
                m2, a2 = rows[j]
                records.append(
                    {
                        'Abstract_Index': abstract_idx,
                        'Target_ID': target_id,
                        'Affinity_Level': level,
                        'Model_1': str(m1),
                        'Model_2': str(m2),
                        'Affinity_Difference': abs(float(a1) - float(a2)),
                    }
                )

    pairwise = pd.DataFrame(records)
    if pairwise.empty:
        return pairwise, pairwise

    pairwise_summary = (
        pairwise.dropna(subset=['Affinity_Level'])
        .groupby(['Affinity_Level', 'Model_1', 'Model_2'], as_index=False)
        .agg(mean_diff=('Affinity_Difference', 'mean'), median_diff=('Affinity_Difference', 'median'), std_diff=('Affinity_Difference', 'std'), n_pairs=('Affinity_Difference', 'count'))
        .sort_values(['Affinity_Level', 'mean_diff'], ascending=[True, False])
    )
    return pairwise, pairwise_summary

In [ ]:
prp_coherency = compute_unit_coherency(prp_df, target_col='PRP_Name')

print("PRP Coherency Summary:")
print(f"Mean CV across all abstract-PRP pairs: {prp_coherency['cv'].mean():.3f}")
print(f"Abstracts with perfect agreement (CV=0): {(prp_coherency['cv'] == 0).sum()}")
print("\nBottom 10 coherent (high CV) abstract-PRP pairs:")
display(prp_coherency.nlargest(10, 'cv')[['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'range']])
print("\nTop 10 coherent (low CV) abstract-PRP pairs:")
display(prp_coherency.nsmallest(10, 'cv')[['Abstract_Index', 'PRP_Name', 'mean_affinity', 'cv', 'range']])

In [ ]:
prp_coherency

In [ ]:
prp_coherency = compute_unit_coherency(prp_df, target_col='PRP_Name')
ra_coherency = compute_unit_coherency(ra_df, target_col='RA2025_ID')

In [ ]:
# Visualization: Coherency Distribution
fig = px.histogram(
    prp_coherency[prp_coherency['cv'].notna()],
    x='cv',
    # nbins=30,
    # facet_col = 'PRP_Name',
    title='PRP Coherency Distribution (Coefficient of Variation)',
    labels={'cv': 'Coefficient of Variation', 'count': 'Number of Pairs'}
)
fig.add_vline(x=prp_coherency['cv'].mean(), line_dash="dash", line_color="red", 
              annotation_text=f"Mean CV: {prp_coherency['cv'].mean():.3f}")

fig.update_xaxes(range=[0,2])
fig.show()

fig = px.histogram(
    ra_coherency[ra_coherency['cv'].notna()],
    x='cv',
    nbins=30,
    title='RA Coherency Distribution (Coefficient of Variation)',
    labels={'cv': 'Coefficient of Variation', 'count': 'Number of Pairs'}
)
fig.add_vline(x=ra_coherency['cv'].mean(), line_dash="dash", line_color="red",
              annotation_text=f"Mean CV: {ra_coherency['cv'].mean():.3f}")
fig.show()

In [ ]:
# Abstract-level coherency scatter plots
fig = px.box(
    prp_coherency,
    x='Abstract_Index',
    y='cv',
    title='PRP Abstract-level Coherency (Lower CV = Higher Agreement)',
    points ='all',
)
fig.update_traces(boxmean=True)  # set to 'sd' to show mean + std
fig.show()

fig = px.box(
    ra_coherency,
    x='Abstract_Index',
    y='cv',
    title='RA Abstract-level Coherency (Lower CV = Higher Agreement)',
)
fig.update_traces(boxmean=True)  # set to 'sd' to show mean + std
fig.show()
